# SAR–Optical Forest Disturbance and Recovery Monitoring Pipeline
## Pan-India Himalayan Region | 2017–2024

**Project:** Low-Cost Remote Sensing Framework for Monitoring Forest Disturbance and Recovery Across the Pan-India Himalayan Region Using SAR–Optical Satellite Time Series

**Study Area:** Jammu & Kashmir · Ladakh · Himachal Pradesh · Uttarakhand · Sikkim · Arunachal Pradesh

**Datasets:** Sentinel-2 SR Harmonized · Sentinel-1 GRD · SRTM DEM · Hansen GFC · GADM 4.1

---
## Workflow Overview — Read Before Running

```
PHASE 1 — EXPORT  (Cells 1–5)
  Build annual fused composites and export to GEE Assets.
  ⏳ Wait for all tasks to complete at https://code.earthengine.google.com/tasks

PHASE 2 — ANALYSIS  (Cells 6–19)
  Load flat assets → disturbance detection → frequency →
  hotspots → recovery modeling → validation → maps → export.
```

**Why two phases?** The Pan-Himalayan AOI spans ~600,000 km². Evaluating deeply-chained
lazy GEE collections simultaneously exceeds the per-user memory limit. Exporting
composites as flat Assets breaks the computation graph; downstream cells operate on
simple rasters.



## Cell 1 — Environment Setup & Authentication

In [ ]:
# ============================================================
# Cell 1 — Environment Setup & GEE Authentication
# ============================================================

import ee
import geemap
import os
import zipfile
import json as _json
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────
GEE_PROJECT    = 'your project id'          # ← replace with your GEE project ID
ASSET_DIR      = f'projects/{GEE_PROJECT}/assets/himalayan_forest2'

EXPORT_SCALE   = 100    # metres — composite export resolution
ANALYSIS_SCALE = 100    # metres — disturbance / recovery analysis
STATS_SCALE    = 5000   # metres — lightweight .getInfo() verification
# ──────────────────────────────────────────────────────────────

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

print(f'GEE initialised  |  project : {GEE_PROJECT}')
print(f'Asset directory  : {ASSET_DIR}')
print(f'Export scale     : {EXPORT_SCALE} m')



## Cell 2 — GADM Diagnostic (Run Once to Identify Correct Shapefile)

In [ ]:
# ============================================================
# Cell 2 — GADM Diagnostic
# ============================================================
# Reads every shapefile inside gadm41_IND_shp.zip and prints
# column names + sample rows so we can confirm which file is
# Level-1 (state boundaries).
# Expected: gadm41_IND_1.shp has ~36 features with NAME_1 column
# ============================================================

ZIP_PATH = 'gadm41_IND_shp.zip'   # ← must be in same folder as notebook

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'{ZIP_PATH} not found.\n'
        f'Files in folder: {os.listdir(".")}'
    )

# List zip contents
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()

shp_files = sorted([f for f in all_files if f.endswith('.shp')])
print(f'Shapefiles in zip: {shp_files}\n')
print('=' * 65)

for shp in shp_files:
    print(f'\nFILE : {shp}')
    print('-' * 65)
    try:
        gdf_tmp = gpd.read_file(f'zip://{ZIP_PATH}!{shp}')
        print(f'  Features : {len(gdf_tmp)}')
        print(f'  Columns  : {list(gdf_tmp.columns)}')
        print(f'  First 3 rows:')
        print(gdf_tmp.drop(columns='geometry').head(3).to_string(index=False))
        name_cols = [c for c in gdf_tmp.columns if 'name' in c.lower()]
        for nc in name_cols[:2]:
            vals = sorted(gdf_tmp[nc].dropna().unique().tolist())
            print(f'  [{nc}] sample: {vals[:8]}')
    except Exception as e:
        print(f'  zip:// read failed: {e}')
        print('  Trying extraction method...')
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                z.extractall('gadm_extracted')
            gdf_tmp = gpd.read_file(f'gadm_extracted/{shp}')
            print(f'  Features : {len(gdf_tmp)}')
            print(f'  Columns  : {list(gdf_tmp.columns)}')
            print(gdf_tmp.drop(columns='geometry').head(3).to_string(index=False))
        except Exception as e2:
            print(f'  Also failed: {e2}')

print('\n' + '=' * 65)
print('Look for file where features ≈ 36 and NAME_1 has state names.')
print('That is your Level-1 file. Cell 3 uses it automatically.')



## Cell 3 — Study Area, Forest Mask & Ancillary Data

In [ ]:
# ============================================================
# Cell 3 — Study Area, Forest Mask & Ancillary Data
# ============================================================
# Reads Level-1 state boundaries directly from the zip,
# filters to 6 Himalayan states, converts to ee.Geometry.
# Applies elevation mask ≥500 m to restrict AOI to Himalayan terrain.
# ============================================================

ZIP_PATH = 'gadm41_IND_shp.zip'

# ── Auto-detect Level-1 shapefile inside zip ──────────────────
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()

level1_shp = next((f for f in all_files if f.endswith('_1.shp')), None)
if level1_shp is None:
    raise FileNotFoundError(
        'No Level-1 .shp found in zip.\n'
        f'All .shp files: {[f for f in all_files if f.endswith(".shp")]}'
    )

print(f'Using Level-1 file: {level1_shp}')

# ── Read shapefile ────────────────────────────────────────────
try:
    gdf = gpd.read_file(f'zip://{ZIP_PATH}!{level1_shp}')
    print('Read using zip:// path')
except Exception:
    print('zip:// failed — extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('gadm_extracted')
    gdf = gpd.read_file(f'gadm_extracted/{level1_shp}')
    print('Read after extraction')

if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

print(f'Total features    : {len(gdf)}')
print(f'Columns           : {list(gdf.columns)}')

# ── Find state name column ────────────────────────────────────
name_col = None
for candidate in ['NAME_1', 'name_1', 'VARNAME_1']:
    if candidate in gdf.columns:
        name_col = candidate
        break
if name_col is None:
    name_col = next(
        (c for c in gdf.columns if 'name' in c.lower() and c != 'geometry'),
        None
    )
if name_col is None:
    raise KeyError(f'State name column not found. Columns: {list(gdf.columns)}')

print(f'State name column : {name_col}')
print(f'All state names   : {sorted(gdf[name_col].unique().tolist())}')

# ── Himalayan states ──────────────────────────────────────────
himalayan_states = [
    'Jammu and Kashmir',
    'Ladakh',
    'Himachal Pradesh',
    'Uttarakhand',
    'Sikkim',
    'Arunachal Pradesh'
]

himalayan_gdf = gdf[gdf[name_col].isin(himalayan_states)].copy()

# Case-insensitive fallback
if len(himalayan_gdf) < len(himalayan_states):
    print(f'\n⚠ Exact match: {len(himalayan_gdf)}/{len(himalayan_states)}. Trying case-insensitive...')
    states_lower  = [s.lower() for s in himalayan_states]
    himalayan_gdf = gdf[gdf[name_col].str.lower().isin(states_lower)].copy()

matched = sorted(himalayan_gdf[name_col].tolist())
missing = set(himalayan_states) - set(himalayan_gdf[name_col].tolist())

print(f'\nStates matched    : {len(himalayan_gdf)}/{len(himalayan_states)}')
print(f'Matched names     : {matched}')
if missing:
    print(f'⚠ Missing         : {missing}')
    print('Update himalayan_states list to match exact spelling shown above.')

# ── Dissolve → ee.Geometry ────────────────────────────────────
dissolved         = himalayan_gdf.dissolve()
aoi_geojson       = _json.loads(dissolved.geometry.to_json())
geom_geojson      = aoi_geojson['features'][0]['geometry']
himalayan_aoi_raw = ee.Geometry(geom_geojson, proj='EPSG:4326', evenOdd=True)

print('\n✓ himalayan_aoi_raw created from GADM 4.1 Level-1')

# ── Elevation mask ≥500 m ─────────────────────────────────────
srtm_raw      = ee.Image('USGS/SRTMGL1_003')
himalaya_mask = srtm_raw.select('elevation').gte(500).clip(himalayan_aoi_raw)
himalayan_aoi = himalaya_mask.updateMask(himalaya_mask).geometry()
print('✓ Elevation mask ≥500 m applied')

# ── Forest mask — Hansen GFC ──────────────────────────────────
# ≥30% tree cover in 2000 AND no loss before study period (2017)
hansen         = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
tree_cover2000 = hansen.select('treecover2000')
loss_year      = hansen.select('lossyear')   # 0=no loss; 1=2001; 17=2017

forest_mask = (
    tree_cover2000.gte(30)
    .And(loss_year.lt(17).Not().Or(loss_year.eq(0)))
    .clip(himalayan_aoi)
    .rename('forest_mask')
    .selfMask()
)
print('✓ Forest mask created (Hansen GFC ≥30% tree cover, no pre-2017 loss)')

# ── SRTM DEM ──────────────────────────────────────────────────
srtm      = srtm_raw.clip(himalayan_aoi)
elevation = srtm.select('elevation')
slope     = ee.Terrain.slope(srtm)
aspect    = ee.Terrain.aspect(srtm)

# ── Time periods ──────────────────────────────────────────────
years             = list(range(2017, 2025))
disturbance_years = list(range(2018, 2025))

print(f'\nStudy area      : Pan-India Himalayan Region')
print(f'AOI source      : {ZIP_PATH} → {level1_shp}')
print(f'State column    : {name_col}')
print(f'States          : {", ".join(himalayan_states)}')
print( 'Elevation filter: ≥500 m (SRTM)')
print( 'Forest mask     : Hansen GFC ≥30% tree cover, no pre-2017 loss')
print(f'Composite years : {years}')
print(f'Detection years : {disturbance_years}')


## Cell 4 — Preprocessing Functions (Do Not Modify)

In [ ]:
# ============================================================
# Cell 4 — Preprocessing Functions (DO NOT MODIFY)
# ============================================================

def mask_s2_clouds(image):
    """
    Mask clouds and cirrus in Sentinel-2 SR using QA60 bitmask.
      Bit 10 — opaque cloud
      Bit 11 — cirrus
    Scales reflectance from DN to [0, 1].
    """
    qa              = image.select('QA60')
    cloud_bit_mask  = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )
    return image.updateMask(mask).divide(10000)


def add_ndsi_and_mask_snow(image):
    """
    Compute NDSI = (Green − SWIR1) / (Green + SWIR1) [B3, B11].
    Mask pixels where NDSI ≥ 0.4 (snow / ice).
    """
    ndsi      = image.normalizedDifference(['B3', 'B11']).rename('NDSI')
    snow_mask = ndsi.lt(0.4)
    return image.updateMask(snow_mask).addBands(ndsi)


def add_vegetation_indices(image):
    """
    Append vegetation and moisture indices:
      NDVI = (B8 − B4)  / (B8 + B4)    canopy greenness
      NBR  = (B8 − B12) / (B8 + B12)   burn / biomass loss sensitivity
      NDMI = (B8 − B11) / (B8 + B11)   canopy moisture
    """
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    nbr  = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
    return image.addBands([ndvi, nbr, ndmi])


print('Preprocessing functions loaded:')
print('  mask_s2_clouds | add_ndsi_and_mask_snow | add_vegetation_indices')


---
## Cell 5 — Composite Builder Functions

In [ ]:
# ============================================================
# Cell 5 — Composite Builder Functions
# ============================================================
# Two optical composite strategies:
#   Standard (2019–2024): Jun–Sep | 60% cloud cap | median
#   Robust  (2017–2018) : Apr–Nov | 85% cloud cap | mean
#     Reason: S2 archive was sparse over Himalayan AOI in
#     early operational years (2017=62 px, 2018=154 px at 10 km).
#     Wider window + mean recovers valid pixels.
# ============================================================

# ── Sentinel-1 base collection ────────────────────────────────
s1_base = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(himalayan_aoi)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['VV', 'VH'])
)


def add_vv_vh_ratio(image):
    """VV/VH ratio in dB-space: ratio_dB = VV_dB − VH_dB."""
    ratio = image.select('VV').subtract(image.select('VH')).rename('VV_VH_ratio')
    return image.addBands(ratio)


def build_optical_composite_standard(year):
    """Jun–Sep median composite — 2019–2024. Output: NDVI, NBR, NDMI."""
    start = f'{year}-06-01'
    end   = f'{year}-09-30'
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(himalayan_aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
        .map(mask_s2_clouds)
        .map(add_ndsi_and_mask_snow)
        .map(add_vegetation_indices)
        .select(['NDVI', 'NBR', 'NDMI'])
        .median()
        .clip(himalayan_aoi)
        .set('year', year)
    )

def build_optical_composite_robust(year):
    """Apr–Nov mean composite — 2017–2018 (sparse S2 archive). Output: NDVI, NBR, NDMI."""
    start = f'{year}-04-01'
    end   = f'{year}-11-30'
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(himalayan_aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 85))
        .map(mask_s2_clouds)
        .map(add_ndsi_and_mask_snow)
        .map(add_vegetation_indices)
        .select(['NDVI', 'NBR', 'NDMI'])
        .mean()
        .clip(himalayan_aoi)
        .set('year', year)
    )


def build_sar_composite(year):
    """Sentinel-1 annual median composite (Jan–Dec). Output: VH, VV_VH_ratio."""
    start = f'{year}-01-01'
    end   = f'{year}-12-31'
    return (
        s1_base
        .filterDate(start, end)
        .map(add_vv_vh_ratio)
        .select(['VH', 'VV_VH_ratio'])
        .median()
        .clip(himalayan_aoi)
        .set('year', year)
    )


SPARSE_YEARS = [2017, 2018]


def build_fused_composite(year):
    """
    SAR–Optical fusion. Auto-selects robust vs standard optical builder.
    Output bands: NDVI, NBR, NDMI, VH, VV_VH_ratio (Float32).
    """
    opt = build_optical_composite_robust(year) if year in SPARSE_YEARS \
          else build_optical_composite_standard(year)
    sar = build_sar_composite(year)
    return (
        opt
        .addBands(sar.select(['VH', 'VV_VH_ratio']))
        .toFloat()
        .set('year', year)
        .set('system:time_start', ee.Date(f'{year}-01-01').millis())
    )

print('Composite builder functions defined.')
print(f'Sparse years (robust strategy): {SPARSE_YEARS}')
print('All other years: standard strategy (Jun–Sep, 60%, median)')


---
## Cell 6 — Export Annual Fused Composites to GEE Assets
### ⚠️ PHASE 1 — Run, then wait for all tasks to complete before continuing

In [ ]:
# ============================================================
# Cell 6 — Export Annual Fused Composites to GEE Assets
# ============================================================
# REQUIRED: Create asset folder first if it does not exist:
#   https://code.earthengine.google.com/ → Assets → New Folder
#   Path: projects/ntr-fms/assets/himalayan_forest2
#
# Set OVERWRITE = True to delete and re-export existing assets.
# ============================================================

OVERWRITE = True

print(f'Submitting export tasks (overwrite={OVERWRITE}) ...')
print(f'Asset directory : {ASSET_DIR}\n')

export_tasks = {}

for yr in years:
    asset_id = f'{ASSET_DIR}/fused_{yr}'
    strategy = 'robust (Apr–Nov, 85%, mean)' if yr in SPARSE_YEARS \
               else 'standard (Jun–Sep, 60%, median)'

    if OVERWRITE:
        try:
            ee.data.deleteAsset(asset_id)
        except Exception:
            pass

    fused = build_fused_composite(yr)

    task = ee.batch.Export.image.toAsset(
        image       = fused,
        description = f'fused_composite_{yr}',
        assetId     = asset_id,
        region      = himalayan_aoi,
        scale       = EXPORT_SCALE,
        maxPixels   = 1e11,
        crs         = 'EPSG:4326'
    )
    task.start()
    export_tasks[yr] = task
    print(f'  {yr}  [{strategy}]')
    print(f'       → {asset_id}')
    print(f'       Task ID: {task.id}')

print(f'\n✓ {len(export_tasks)} tasks submitted.')
print('━' * 65)
print('NEXT STEP: Wait for ALL tasks COMPLETED, then run Cell 7.')
print('Monitor: https://code.earthengine.google.com/tasks')


---
## ⏳ Wait — All `fused_YYYY` assets must show **COMPLETED** before continuing
https://code.earthengine.google.com/tasks

---
## PHASE 2 — Analysis

---
## Cell 7 — Load Assets & Verify Coverage

In [ ]:
# ============================================================
# Cell 7 — Load Fused Composites from Assets & Verify Coverage
# ============================================================
# Loading from assets gives GEE flat rasters with no upstream
# lazy chain — eliminates the memory limit error.
# Expected: all years ≥ 1,000 valid NDVI pixels at 10 km.
# ============================================================

def load_fused_asset(year):
    """Load a pre-exported fused composite asset."""
    return (
        ee.Image(f'{ASSET_DIR}/fused_{year}')
        .set('year', year)
        .set('system:time_start', ee.Date(f'{year}-01-01').millis())
    )


annual_optical_list = []
annual_sar_list     = []
fused_list          = []

for yr in years:
    img = load_fused_asset(yr)
    annual_optical_list.append(img.select(['NDVI', 'NBR', 'NDMI']).set('year', yr))
    annual_sar_list.append(img.select(['VH', 'VV_VH_ratio']).set('year', yr))
    fused_list.append(img)

annual_optical   = ee.ImageCollection(annual_optical_list)
annual_sar       = ee.ImageCollection(annual_sar_list)
fused_composites = ee.ImageCollection(fused_list)

# ── Band structure check ──────────────────────────────────────
sample_bands = fused_composites.filter(ee.Filter.eq('year', 2020)).first().bandNames().getInfo()
print(f'Bands : {sample_bands}')
assert set(sample_bands) == {'NDVI', 'NBR', 'NDMI', 'VH', 'VV_VH_ratio'}, \
    'Band mismatch — re-check export'

# ── Per-year coverage check ───────────────────────────────────
print('\nCoverage check (valid NDVI pixels at 10 km scale):\n')
coverage_ok = True

for yr in years:
    img = fused_composites.filter(ee.Filter.eq('year', yr)).first()
    n = int((img.select('NDVI').mask()
             .reduceRegion(ee.Reducer.sum(), himalayan_aoi, 10000,
                           maxPixels=1e8, bestEffort=True)
             .get('NDVI').getInfo()) or 0)
    v = int((img.select('VH').mask()
             .reduceRegion(ee.Reducer.sum(), himalayan_aoi, 10000,
                           maxPixels=1e8, bestEffort=True)
             .get('VH').getInfo()) or 0)
    flag = ' ⚠️  SPARSE — re-export' if n < 500 else ' ✓'
    if n < 500:
        coverage_ok = False
    print(f'  {yr} — NDVI: {n:>6,} px  |  VH: {v:>6,} px{flag}')

print()
if coverage_ok:
    print('✓ All years adequate. Continue to Cell 8.')
else:
    print('⚠ Sparse years found. Re-run Cell 6 for flagged years, then reload.')


---
## Cell 8 — Disturbance Detection Function

In [ ]:
# ============================================================
# Cell 8 — Disturbance Detection Function
# ============================================================
# Method: bi-temporal SAR–Optical change detection
#
# Change indices (current minus previous year):
#   dNBR  = NBR(t)  − NBR(t−1)   fire / biomass loss
#   dNDVI = NDVI(t) − NDVI(t−1)  canopy greenness loss
#   dNDMI = NDMI(t) − NDMI(t−1)  canopy moisture loss
#   dVH   = VH(t)   − VH(t−1)    structural canopy loss (SAR)
#
# Disturbance thresholds:
#   Optical : dNBR < −0.15 OR dNDVI < −0.07 OR dNDMI < −0.05
#   SAR     : dVH  < −1.5 dB
#   Final   : optical_dist OR sar_dist
#
# Noise filter: connectedPixelCount > 10 (8-connectivity)
# Output: band 'disturbance' (Byte) — 1=disturbed, masked=undisturbed
# ============================================================

def compute_disturbance(year):
    img_curr = fused_composites.filter(ee.Filter.eq('year', year)).first()
    img_prev = fused_composites.filter(ee.Filter.eq('year', year - 1)).first()

    dNBR  = img_curr.select('NBR').subtract(img_prev.select('NBR'))
    dNDVI = img_curr.select('NDVI').subtract(img_prev.select('NDVI'))
    dNDMI = img_curr.select('NDMI').subtract(img_prev.select('NDMI'))
    dVH   = img_curr.select('VH').subtract(img_prev.select('VH'))

    optical_dist = dNBR.lt(-0.15).Or(dNDVI.lt(-0.07)).Or(dNDMI.lt(-0.05))
    sar_dist     = dVH.lt(-1.5)

    disturbance_raw    = optical_dist.Or(sar_dist)
    disturbance_forest = disturbance_raw.updateMask(forest_mask)

    connected         = disturbance_forest.connectedPixelCount(maxSize=128, eightConnected=True)
    disturbance_clean = disturbance_forest.updateMask(connected.gt(10))

    return (
        disturbance_clean
        .rename('disturbance')
        .toByte()
        .set('year', year)
        .set('system:time_start', ee.Date(f'{year}-01-01').millis())
    )


print('compute_disturbance() defined.')
print('Logic  : (dNBR<−0.15 OR dNDVI<−0.07 OR dNDMI<−0.05) OR (dVH<−1.5 dB)')
print('Filter : connectedPixelCount > 10  |  Hansen forest mask applied')


---
## Cell 9 — Annual Disturbance Collection (2018–2024)

In [ ]:
# ============================================================
# Cell 9 — Build Annual Disturbance Collection (2018–2024)
# ============================================================

annual_disturbance_list = [compute_disturbance(yr) for yr in disturbance_years]
annual_disturbances     = ee.ImageCollection(annual_disturbance_list)

print(f'Annual disturbance collection : {disturbance_years}')
print(f'Images in collection          : {annual_disturbances.size().getInfo()}')

print('\nPer-year disturbance pixel count (5 km scale, bestEffort):\n')
for yr in disturbance_years:
    img   = annual_disturbances.filter(ee.Filter.eq('year', yr)).first()
    count = (img.reduceRegion(
                 reducer=ee.Reducer.sum(), geometry=himalayan_aoi,
                 scale=STATS_SCALE, maxPixels=1e8, bestEffort=True
             ).get('disturbance').getInfo())
    print(f'  {yr} : {int(count or 0):>8,} pixels disturbed')


---
## Cell 10 — Disturbance Year Map

In [ ]:
# ============================================================
# Cell 10 — Disturbance Year Map
# ============================================================
# Assigns the EARLIEST disturbance year to each forest pixel.
# .toInt16() explicit cast prevents Short<0,2018>/Short<0,2019>
# type mismatch error in GEE ImageCollection.
# ============================================================

def assign_year(image):
    year_val = ee.Number(image.get('year'))
    return (
        image.select('disturbance')
        .multiply(year_val)
        .toInt16()        # explicit cast — fixes homogeneity error
        .selfMask()
        .rename('dist_year')
        .set('year', year_val)
    )


disturbance_year_map = (
    annual_disturbances
    .map(assign_year)
    .select('dist_year')
    .min()
    .clip(himalayan_aoi)
    .rename('disturbance_year')
)

dist_stats = disturbance_year_map.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=himalayan_aoi,
    scale=STATS_SCALE, maxPixels=1e8, bestEffort=True
).getInfo()

print('Disturbance year map created (earliest disturbance year per pixel).')
print(f'Pixel-value range (5 km scale): {dist_stats}')


---
## Cell 11 — Disturbance Frequency

In [ ]:
# ============================================================
# Cell 11 — Disturbance Frequency
# ============================================================
# Counts how many years (out of 2018–2024) each forest pixel
# was classified as disturbed. Range: 0–7.
# ============================================================

disturbance_frequency = (
    annual_disturbances
    .select('disturbance')
    .sum()
    .toInt16()
    .clip(himalayan_aoi)
    .updateMask(forest_mask)
    .rename('disturbance_frequency')
)

freq_stats = disturbance_frequency.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(), geometry=himalayan_aoi,
    scale=STATS_SCALE, maxPixels=1e8, bestEffort=True
).getInfo()

print('Disturbance frequency map created.')
print('Histogram (events : pixel count at 5 km scale):\n')
hist = freq_stats.get('disturbance_frequency', {})
for k in sorted(hist.keys(), key=lambda x: float(x)):
    print(f'  {int(float(k))} event(s) : {int(hist[k]):>8,} pixels')


---
## Cell 12 — Disturbance Hotspot Classification

In [ ]:
# ============================================================
# Cell 12 — Disturbance Hotspot Classification
# ============================================================
#   Class 1 — Low disturbance      : 1–2 events
#   Class 2 — Moderate disturbance : 3–4 events
#   Class 3 — High disturbance     : ≥5 events
# ============================================================

freq = disturbance_frequency

low_dist  = freq.gte(1).And(freq.lte(2)).multiply(1)
mod_dist  = freq.gte(3).And(freq.lte(4)).multiply(2)
high_dist = freq.gte(5).multiply(3)

hotspot_map = (
    low_dist.add(mod_dist).add(high_dist)
    .selfMask().toByte()
    .clip(himalayan_aoi)
    .updateMask(forest_mask)
    .rename('hotspot_class')
)

hs_stats = hotspot_map.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(), geometry=himalayan_aoi,
    scale=STATS_SCALE, maxPixels=1e8, bestEffort=True
).getInfo()

labels = {'1': 'Low (1–2)', '2': 'Moderate (3–4)', '3': 'High (≥5)'}
print('Hotspot classification complete:\n')
for k in sorted(hs_stats.get('hotspot_class', {}).keys(), key=lambda x: float(x)):
    ki = str(int(float(k)))
    print(f'  Class {ki} [{labels.get(ki, ki)}] : {int(hs_stats["hotspot_class"][k]):>8,} pixels')


---
## Cell 13 — Recovery Trajectory Modeling

In [ ]:
# ============================================================
# Cell 13 — Recovery Trajectory Modeling
# ============================================================
# Model: NDVI(t) = a + b·t  where t = years since 2017
#   b = recovery slope (yr⁻¹)
# GEE linearFit() : band[0]=time (X), band[1]=NDVI or VH (Y)
# Returns: 'scale' (slope b), 'offset' (intercept a)
# ============================================================

recovery_years = list(range(2019, 2025))


def make_recovery_image(year):
    ndvi = annual_optical.filter(ee.Filter.eq('year', year)).first().select('NDVI')
    vh   = annual_sar.filter(ee.Filter.eq('year', year)).first().select('VH')
    t    = ee.Image.constant(ee.Number(year).subtract(2017)).toFloat().rename('time')
    prior_dist = disturbance_year_map.lt(year).And(disturbance_year_map.gt(0))
    return (
        t.addBands(ndvi.toFloat()).addBands(vh.toFloat())
        .updateMask(prior_dist)
        .updateMask(forest_mask)
        .set('year', year)
        .set('system:time_start', ee.Date(f'{year}-01-01').millis())
    )


recovery_stack = ee.ImageCollection([make_recovery_image(yr) for yr in recovery_years])

# ── NDVI recovery slope ───────────────────────────────────────
ndvi_fit = recovery_stack.select(['time', 'NDVI']).reduce(ee.Reducer.linearFit())
recovery_rate_map = ndvi_fit.select('scale').clip(himalayan_aoi).rename('recovery_rate')

# ── SAR structural recovery slope ────────────────────────────
sar_fit = recovery_stack.select(['time', 'VH']).reduce(ee.Reducer.linearFit())
sar_recovery_map = sar_fit.select('scale').clip(himalayan_aoi).rename('sar_recovery_rate')

# ── Recovery time estimate ────────────────────────────────────
ndvi_baseline = (annual_optical.filter(ee.Filter.eq('year', 2017)).first()
                 .select('NDVI').updateMask(forest_mask))
ndvi_2024     = annual_optical.filter(ee.Filter.eq('year', 2024)).first().select('NDVI')
ndvi_deficit  = (ndvi_baseline.subtract(ndvi_2024)
                 .updateMask(disturbance_year_map.gt(0)).updateMask(forest_mask))
recovery_time_map = (
    ndvi_deficit
    .divide(recovery_rate_map.max(ee.Image.constant(0.001)))
    .max(ee.Image.constant(0))
    .clip(himalayan_aoi)
    .rename('recovery_time_years')
)

print('Recovery trajectory modeling complete.')
print('  recovery_rate_map  — NDVI slope b (yr⁻¹)')
print('  sar_recovery_map   — VH slope (dB yr⁻¹)')
print('  recovery_time_map  — estimated years to baseline NDVI')


---
## Cell 14 — Recovery Classification

In [ ]:
# ============================================================
# Cell 14 — Recovery Classification
# ============================================================
#   Class 1 — No recovery       : NDVI slope ≤ 0
#   Class 2 — Slow recovery     : 0 < slope ≤ 0.005
#   Class 3 — Moderate recovery : 0.005 < slope ≤ 0.02
#   Class 4 — Fast recovery     : slope > 0.02
# ============================================================

slope = recovery_rate_map

no_recovery   = slope.lte(0.000).multiply(1)
slow_recovery = slope.gt(0.000).And(slope.lte(0.005)).multiply(2)
mod_recovery  = slope.gt(0.005).And(slope.lte(0.020)).multiply(3)
fast_recovery = slope.gt(0.020).multiply(4)

recovery_class_map = (
    no_recovery.add(slow_recovery).add(mod_recovery).add(fast_recovery)
    .toByte()
    .clip(himalayan_aoi)
    .updateMask(disturbance_year_map.gt(0))
    .updateMask(forest_mask)
    .rename('recovery_class')
)

rc_stats = recovery_class_map.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(), geometry=himalayan_aoi,
    scale=STATS_SCALE, maxPixels=1e8, bestEffort=True
).getInfo()

rc_labels = {'1':'No recovery (≤0)','2':'Slow (0–0.005)',
             '3':'Moderate (0.005–0.02)','4':'Fast (>0.02)'}
print('Recovery classification complete:\n')
for k in sorted(rc_stats.get('recovery_class', {}).keys(), key=lambda x: float(x)):
    ki = str(int(float(k)))
    print(f'  Class {ki} [{rc_labels.get(ki,ki)}] : {int(rc_stats["recovery_class"][k]):>8,} pixels')


---
## Cell 15 — Interactive Maps (geemap)

In [ ]:
# ============================================================
# Cell 15 — Interactive Maps (geemap)
# ============================================================

# Shared resources
s2_rgb = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(himalayan_aoi)
    .filterDate('2022-06-01', '2022-09-30')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(mask_s2_clouds)
    .select(['B4', 'B3', 'B2'])
    .median()
    .clip(himalayan_aoi)
)
aoi_outline = ee.Image().paint(ee.FeatureCollection(himalayan_aoi), 0, 2)
print('Shared basemap ready.')

# ── Map 1: Disturbance Year ───────────────────────────────────
Map1 = geemap.Map()
Map1.centerObject(himalayan_aoi, zoom=6)
Map1.addLayer(s2_rgb, {'min':0.0,'max':0.3,'gamma':1.4}, 'S2 True Colour 2022')
Map1.addLayer(disturbance_year_map,
    {'min':2018,'max':2024,
     'palette':['#ffffcc','#fed976','#fd8d3c','#f03b20','#bd0026','#800026','#4d0013']},
    'Disturbance Year (2018–2024)')
Map1.addLayer(aoi_outline, {'palette':['red']}, 'AOI Boundary')
Map1.add_colorbar(
    {'min':2018,'max':2024,
     'palette':['#ffffcc','#fed976','#fd8d3c','#f03b20','#bd0026','#800026','#4d0013']},
    label='First Disturbance Year')
print('Map 1: Disturbance Year')
Map1


In [ ]:
# ── Map 2: Disturbance Frequency ─────────────────────────────
Map2 = geemap.Map()
Map2.centerObject(himalayan_aoi, zoom=6)
Map2.addLayer(s2_rgb, {'min':0.0,'max':0.3,'gamma':1.4}, 'S2 True Colour 2022')
Map2.addLayer(disturbance_frequency,
    {'min':1,'max':7,
     'palette':['#ffffb2','#fecc5c','#fd8d3c','#f03b20','#bd0026','#7a0010','#3d0005']},
    'Disturbance Frequency')
Map2.addLayer(aoi_outline, {'palette':['blue']}, 'AOI Boundary')
Map2.add_colorbar(
    {'min':1,'max':7,
     'palette':['#ffffb2','#fecc5c','#fd8d3c','#f03b20','#bd0026','#7a0010','#3d0005']},
    label='Number of Disturbance Events')
print('Map 2: Disturbance Frequency')
Map2

In [ ]:
# ── Map 3: Disturbance Hotspots ──────────────────────────────
Map3 = geemap.Map()
Map3.centerObject(himalayan_aoi, zoom=6)
Map3.addLayer(s2_rgb, {'min':0.0,'max':0.3,'gamma':1.4}, 'S2 True Colour 2022')
Map3.addLayer(hotspot_map,
    {'min':1,'max':3,'palette':['#fee391','#f03b20','#7a0010']},
    'Disturbance Hotspots')
Map3.addLayer(aoi_outline, {'palette':['black']}, 'AOI Boundary')
Map3.add_legend(title='Disturbance Hotspot Class',
    keys=['Low (1–2 events)','Moderate (3–4 events)','High (≥5 events)'],
    colors=['#fee391','#f03b20','#7a0010'])
print('Map 3: Disturbance Hotspots')
Map3

In [ ]:
# ── Map 4: Recovery Classification ───────────────────────────
Map4 = geemap.Map()
Map4.centerObject(himalayan_aoi, zoom=6)
Map4.addLayer(s2_rgb, {'min':0.0,'max':0.3,'gamma':1.4}, 'S2 True Colour 2022')
Map4.addLayer(recovery_class_map,
    {'min':1,'max':4,'palette':['#d73027','#fc8d59','#fee090','#1a9850']},
    'Recovery Classification')
Map4.addLayer(aoi_outline, {'palette':['blue']}, 'AOI Boundary')
Map4.add_legend(title='NDVI Recovery Class',
    keys=['No Recovery (≤0)','Slow (0–0.005)','Moderate (0.005–0.02)','Fast (>0.02)'],
    colors=['#d73027','#fc8d59','#fee090','#1a9850'])
print('Map 4: Recovery Classification')
Map4

In [ ]:
# ── Map 5: NDVI Recovery Rate ────────────────────────────────
Map5 = geemap.Map()
Map5.centerObject(himalayan_aoi, zoom=6)
Map5.addLayer(s2_rgb, {'min':0.0,'max':0.3,'gamma':1.4}, 'S2 True Colour 2022')
Map5.addLayer(recovery_rate_map,
    {'min':-0.05,'max':0.05,
     'palette':['#d73027','#fc8d59','#ffffbf','#91cf60','#1a9850']},
    'NDVI Recovery Rate (slope yr⁻¹)')
Map5.addLayer(aoi_outline, {'palette':['black']}, 'AOI Boundary')
Map5.add_colorbar(
    {'min':-0.05,'max':0.05,
     'palette':['#d73027','#fc8d59','#ffffbf','#91cf60','#1a9850']},
    label='NDVI Trend (yr⁻¹)')
print('Map 5: NDVI Recovery Rate')
Map5

---
## Cell 16 — Matplotlib Summary Figures

In [ ]:
# ============================================================
# Cell 16 — Matplotlib Summary Figures
# ============================================================
# get_hist_val() safely handles GEE histogram key formats:
#   '1', '1.0', or 1 — all tried to avoid empty charts.
# ============================================================

def get_hist_val(hist, key):
    """Safely retrieve histogram value regardless of key format."""
    return int(hist.get(str(key)) or hist.get(str(float(key))) or hist.get(key) or 0)


# ── Annual disturbed area ─────────────────────────────────────
print('Fetching annual disturbance areas ...')
annual_area_km2 = []
for yr in disturbance_years:
    dist_img = annual_disturbances.filter(ee.Filter.eq('year', yr)).first().select('disturbance')
    val = (dist_img.multiply(ee.Image.pixelArea().divide(1e6))
           .reduceRegion(ee.Reducer.sum(), himalayan_aoi, STATS_SCALE,
                         maxPixels=1e8, bestEffort=True)
           .get('disturbance').getInfo())
    annual_area_km2.append(float(val) if val else 0.0)
    print(f'  {yr} : {annual_area_km2[-1]:,.0f} km²')

# ── Hotspot counts ────────────────────────────────────────────
hs_raw = (hotspot_map.reduceRegion(ee.Reducer.frequencyHistogram(),
          himalayan_aoi, STATS_SCALE, maxPixels=1e8, bestEffort=True)
          .getInfo().get('hotspot_class', {}))
hs_counts = [get_hist_val(hs_raw, k) for k in [1, 2, 3]]
print(f'\nHotspot counts (Low/Mod/High): {hs_counts}')

# ── Recovery counts ───────────────────────────────────────────
rc_raw = (recovery_class_map.reduceRegion(ee.Reducer.frequencyHistogram(),
          himalayan_aoi, STATS_SCALE, maxPixels=1e8, bestEffort=True)
          .getInfo().get('recovery_class', {}))
rc_counts = [get_hist_val(rc_raw, k) for k in [1, 2, 3, 4]]
print(f'Recovery counts (None/Slow/Mod/Fast): {rc_counts}')

# ── Figure ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle(
    'SAR–Optical Forest Disturbance & Recovery\nPan-India Himalayan Region  |  2018–2024',
    fontsize=14, fontweight='bold'
)

# Plot 1: Annual area
ax = axes[0]
bars1 = ax.bar(disturbance_years, annual_area_km2,
               color='#bd0026', edgecolor='#7a0010', linewidth=0.8)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Disturbed Forest Area (km²)', fontsize=11)
ax.set_title('Annual Forest Disturbance Area', fontsize=12)
ax.set_xticks(disturbance_years)
ax.set_xticklabels(disturbance_years, rotation=45)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars1, annual_area_km2):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=8)

# Plot 2: Hotspot pie
ax = axes[1]
nz = [(v,l,c,e) for v,l,c,e in
      zip(hs_counts,
          ['Low\n(1–2)','Moderate\n(3–4)','High\n(≥5)'],
          ['#fee391','#f03b20','#7a0010'],
          [0, 0.05, 0.10]) if v > 0]
if nz:
    vs,ls,cs,es = zip(*nz)
    ax.pie(vs, labels=ls, colors=cs, explode=es, autopct='%1.1f%%',
           startangle=140, textprops={'fontsize':9})
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center',
            transform=ax.transAxes, fontsize=12, color='gray')
ax.set_title('Disturbance Hotspot Distribution', fontsize=12)

# Plot 3: Recovery bars
ax = axes[2]
bars3 = ax.bar(['No Recovery\n(≤0)','Slow\n(0–0.005)',
                'Moderate\n(0.005–0.02)','Fast\n(>0.02)'],
               rc_counts,
               color=['#d73027','#fc8d59','#fee090','#1a9850'],
               edgecolor='gray', linewidth=0.6)
ax.set_ylabel('Pixel Count (5 km resolution)', fontsize=11)
ax.set_title('Post-Disturbance Recovery Classification', fontsize=12)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars3, rc_counts):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('himalayan_disturbance_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nFigure saved: himalayan_disturbance_summary.png')


---
## Cell 17 — Disturbance Analysis by Elevation Zone

In [ ]:
# ============================================================
# Cell 17 — Disturbance Analysis by Elevation Zone
# ============================================================
# Elevation zones:
#   1 — Lower Himalaya : 500–1500 m
#   2 — Mid Himalaya   : 1500–2500 m
#   3 — Upper Himalaya : 2500–3500 m
#   4 — Alpine zone    : >3500 m
# ============================================================

print('Computing disturbance statistics by elevation zone...')

zone1 = elevation.gte(500).And(elevation.lt(1500)).multiply(1)
zone2 = elevation.gte(1500).And(elevation.lt(2500)).multiply(2)
zone3 = elevation.gte(2500).And(elevation.lt(3500)).multiply(3)
zone4 = elevation.gte(3500).multiply(4)

elevation_zones = (
    zone1.add(zone2).add(zone3).add(zone4)
    .updateMask(elevation.gte(500))
    .rename('elevation_zone')
)

pixel_area      = ee.Image.pixelArea()
disturbed_pixels = disturbance_frequency.gt(0)

zone_names = {
    1: 'Lower Himalaya (500–1500m)',
    2: 'Mid Himalaya (1500–2500m)',
    3: 'Upper Himalaya (2500–3500m)',
    4: 'Alpine (>3500m)'
}

zone_stats = []
for zone in [1, 2, 3, 4]:
    zone_mask = elevation_zones.eq(zone)
    forest_area = (forest_mask.updateMask(zone_mask).multiply(pixel_area)
                   .reduceRegion(ee.Reducer.sum(), himalayan_aoi,
                                 500, maxPixels=1e10, bestEffort=True).getInfo())
    dist_area = (disturbed_pixels.updateMask(zone_mask).multiply(pixel_area)
                 .reduceRegion(ee.Reducer.sum(), himalayan_aoi,
                               500, maxPixels=1e10, bestEffort=True).getInfo())
    forest_km2 = (forest_area.get('forest_mask') or 0) / 1e6
    dist_km2   = (dist_area.get('disturbance_frequency') or 0) / 1e6
    pct        = (dist_km2 / forest_km2 * 100) if forest_km2 > 0 else 0
    zone_stats.append({
        'Elevation Zone'       : zone_names[zone],
        'Forest Area (km²)'   : round(forest_km2, 2),
        'Disturbed Area (km²)': round(dist_km2, 2),
        'Disturbance (%)'     : round(pct, 2)
    })

elevation_df = pd.DataFrame(zone_stats)
print('\nDisturbance by Elevation Zone')
print('=' * 70)
print(elevation_df.to_string(index=False))


---
# Multi-Source Validation Stage
## Low-Cost Deforestation Detection — Non-Tropical Himalayan Forests

**Validation Design:**
- **Sensor independence**: Hansen GFC (Landsat) vs SAR–Optical model (S1+S2)
- **Circular validation avoided**: Dynamic World (S2-derived) used as exclusion mask ONLY
- **Triple-source forest masking**: Hansen + WorldCover + Dynamic World must agree
- **Study-period aligned**: Hansen lossyear 18–24 matches model detection window

```
Cell 18a  Load validation datasets
Cell 18b  Hansen forest mask
Cell 18c  WorldCover forest mask
Cell 18d  Dynamic World mask (exclusion only)
Cell 18e  Combined triple-source forest mask
Cell 18f  Hansen disturbance reference
Cell 18g  Stratified sampling (10,000 pts @ 30 m)
Cell 18h  Confusion matrix
Cell 18i  Accuracy metrics (OA, Precision, Recall, F1, Kappa)
Cell 18j  Visualisation
```

---
## Validation Cell 18a — Load Validation Datasets

In [ ]:
# ============================================================
# TRIPLE-SOURCE VALIDATION — FIXED VERSION
# Hansen GFC + ALOS PALSAR + MODIS Burned Area
#
# Fix: ALOS PALSAR FNF4 collection availability check +
#      null-safe .first() with .mosaic() fallback +
#      correct date range for each dataset
# ============================================================

import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, classification_report,
                              cohen_kappa_score)

print("=" * 65)
print("  TRIPLE-SOURCE VALIDATION — FIXED")
print("  Hansen GFC | ALOS PALSAR | MODIS Burned Area")
print("=" * 65)

# ── DIAGNOSTIC: Check ALOS PALSAR availability first ──────────
print("\n[DIAGNOSTIC] Checking ALOS PALSAR FNF4 collection...")

alos_col = (
    ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/FNF4')
    .filterBounds(himalayan_aoi)
)
alos_count = alos_col.size().getInfo()
print(f"  Total ALOS PALSAR images in AOI : {alos_count}")

if alos_count > 0:
    # Print available years
    alos_years = (
        alos_col
        .aggregate_array('system:time_start')
        .map(lambda t: ee.Date(t).get('year'))
        .distinct()
        .sort()
        .getInfo()
    )
    print(f"  Available years               : {alos_years}")
else:
    print("  ⚠ No ALOS FNF4 images — will use PALSAR mosaic fallback")

# ── STEP 1A: Hansen GFC (same as before — no changes needed) ──
print("\nStep 1A: Hansen GFC — canopy loss 2018–2024...")

hansen_loss = (
    loss_year
    .clip(himalayan_aoi)
    .gte(18).And(loss_year.clip(himalayan_aoi).lte(24))
    .unmask(0)
    .rename('hansen_loss')
    .toByte()
)
print("  ✓ Hansen GFC loaded")

# ── STEP 1B: ALOS PALSAR — NULL-SAFE loading ──────────────────
print("\nStep 1B: ALOS PALSAR — structural forest change (null-safe)...")

def load_alos_year(year_str):
    """
    Safely load ALOS PALSAR FNF4 for a given year.
    Falls back to nearest available year if exact year missing.
    Returns None image replaced by zeros if collection empty.
    """
    col = (
        ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/FNF4')
        .filterBounds(himalayan_aoi)
        .filterDate(f'{year_str}-01-01', f'{year_str}-12-31')
    )
    size = col.size().getInfo()
    if size > 0:
        img = col.mosaic().select('fnf').clip(himalayan_aoi)
        print(f"    ✓ ALOS {year_str}: {size} image(s) found → using mosaic")
        return img
    else:
        print(f"    ⚠ ALOS {year_str}: no images — trying mosaic of full collection...")
        # Use full collection mosaic clipped to AOI as best estimate
        full = (
            ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/FNF4')
            .filterBounds(himalayan_aoi)
        )
        full_size = full.size().getInfo()
        if full_size > 0:
            img = full.sort('system:time_start').first().select('fnf').clip(himalayan_aoi)
            print(f"    ✓ Using earliest available ALOS image as baseline")
            return img
        else:
            # Complete fallback: zero image (no ALOS contribution)
            print(f"    ✗ ALOS PALSAR unavailable for AOI — using zero fallback")
            return None

alos_baseline = load_alos_year('2017')   # pre-disturbance baseline
alos_endyear  = load_alos_year('2023')   # post-disturbance (latest available)

if alos_baseline is not None and alos_endyear is not None:
    # FNF4 classes: 1=Dense Forest, 2=Non-Dense Forest, 3=Non-Forest, 4=Water
    # Disturbance: was forest (1 or 2) → became non-forest (3) or water (4)
    alos_disturbance = (
        alos_baseline.lte(2)          # was any forest in baseline year
        .And(alos_endyear.gte(3))     # became non-forest by end year
        .unmask(0)
        .rename('alos_disturbance')
        .toByte()
    )
    print("  ✓ ALOS PALSAR disturbance layer created (forest→non-forest)")
else:
    # Zero fallback — ALOS contributes nothing, validation uses Hansen+MODIS only
    alos_disturbance = (
        ee.Image.constant(0)
        .clip(himalayan_aoi)
        .rename('alos_disturbance')
        .toByte()
    )
    print("  ⚠ ALOS fallback: zero image (Hansen+MODIS validation only)")

# ── STEP 1C: MODIS Burned Area — NULL-SAFE loading ────────────
print("\nStep 1C: MODIS MCD64A1 — burned area 2018–2024 (null-safe)...")

modis_col = (
    ee.ImageCollection('MODIS/061/MCD64A1')
    .filterBounds(himalayan_aoi)
    .filterDate('2018-01-01', '2024-12-31')
    .select('BurnDate')
)
modis_count = modis_col.size().getInfo()
print(f"  MODIS images found: {modis_count}")

if modis_count > 0:
    modis_burned = (
        modis_col
        .map(lambda img: img.gt(0).unmask(0).toByte())
        .max()
        .clip(himalayan_aoi)
        .unmask(0)
        .rename('modis_burned')
        .toByte()
    )
    print("  ✓ MODIS Burned Area loaded")
else:
    # Try older version (v006)
    print("  Trying MODIS/006/MCD64A1 fallback...")
    modis_col_v6 = (
        ee.ImageCollection('MODIS/006/MCD64A1')
        .filterBounds(himalayan_aoi)
        .filterDate('2018-01-01', '2024-12-31')
        .select('BurnDate')
    )
    v6_count = modis_col_v6.size().getInfo()
    if v6_count > 0:
        modis_burned = (
            modis_col_v6
            .map(lambda img: img.gt(0).unmask(0).toByte())
            .max()
            .clip(himalayan_aoi)
            .unmask(0)
            .rename('modis_burned')
            .toByte()
        )
        print(f"  ✓ MODIS v006 fallback loaded ({v6_count} images)")
    else:
        modis_burned = (
            ee.Image.constant(0)
            .clip(himalayan_aoi)
            .rename('modis_burned')
            .toByte()
        )
        print("  ⚠ MODIS fallback: zero image")

# ── STEP 2: Combined Reference ────────────────────────────────
print("\nStep 2: Building combined reference (Hansen OR ALOS OR MODIS)...")

combined_reference = (
    hansen_loss
    .Or(alos_disturbance)
    .Or(modis_burned)
    .updateMask(forest_mask)
    .clip(himalayan_aoi)
    .unmask(0)                    # ← KEY FIX: unmask AFTER updateMask
    .rename('combined_reference')
    .toByte()
)

# Per-source stats
def get_dist_count(img, band):
    h = img.updateMask(forest_mask).unmask(0).reduceRegion(
        reducer   = ee.Reducer.frequencyHistogram(),
        geometry  = himalayan_aoi,
        scale     = 5000,
        maxPixels = 1e8,
        bestEffort= True
    ).getInfo().get(band, {})
    def gv(k):
        return int(float(h.get(str(k), h.get(str(float(k)), h.get(k, 0)))))
    return gv(0), gv(1)

print("\n  Per-source disturbed pixel counts (5km scale):")
u_h, d_h = get_dist_count(hansen_loss,      'hansen_loss')
u_a, d_a = get_dist_count(alos_disturbance, 'alos_disturbance')
u_m, d_m = get_dist_count(modis_burned,     'modis_burned')
u_c, d_c = get_dist_count(combined_reference,'combined_reference')

for name, ud, dd in [
    ("Hansen GFC",       u_h, d_h),
    ("ALOS PALSAR",      u_a, d_a),
    ("MODIS Burned",     u_m, d_m),
    ("COMBINED (OR)",    u_c, d_c),
]:
    print(f"    {name:<18}: undist={ud:>7,}  dist={dd:>7,}")

# ── STEP 3: Model Prediction Map ──────────────────────────────
print("\nStep 3: Building model prediction map...")

val_predicted = (
    disturbance_year_map
    .clip(himalayan_aoi)
    .gt(0)
    .unmask(0)
    .updateMask(forest_mask)
    .unmask(0)                    # ← unmask after forest mask
    .rename('predicted')
    .toByte()
)
print("  ✓ Prediction map ready")

# ── STEP 4: Combine & Export ───────────────────────────────────
#print("\nStep 4: Exporting stratified sample to Google Drive...")

val_image_triple = val_predicted.addBands(combined_reference)

val_samples_triple = val_image_triple.stratifiedSample(
    numPoints  = 3000,
    classBand  = 'combined_reference',
    region     = himalayan_aoi,
    scale      = 100,             # match analysis scale
    seed       = 42,
    tileScale  = 8,
    geometries = False
)

"""export_triple = ee.batch.Export.table.toDrive(
    collection     = val_samples_triple,
    description    = 'triple_validation_samples',
    folder         = 'GEE_Exports',
    fileNamePrefix = 'triple_validation_samples',
    fileFormat     = 'CSV',
    selectors      = ['predicted', 'combined_reference']
)#!
export_triple.start()
print(f"  ✓ Export submitted  |  Task ID: {export_triple.id}")
print("  ⏳ Wait for completion at https://code.earthengine.google.com/tasks")
print("  Then download triple_validation_samples.csv and run Part 2 below")"""

In [ ]:
# ============================================================
# PART 2 — Run after downloading triple_validation_samples.csv
# ============================================================

import os

CSV_PATH = 'triple_validation_samples.csv'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"'{CSV_PATH}' not found.\n"
        f"Files in folder: {os.listdir('.')}"
    )

raw = pd.read_csv(CSV_PATH)

# Column name normalisation (handles any GEE export variation)
raw.columns = [c.strip() for c in raw.columns]
pred_col = next(c for c in raw.columns if 'predict' in c.lower())
ref_col  = next(c for c in raw.columns if 'reference' in c.lower()
                or 'combined' in c.lower())

val_df = (
    raw[[pred_col, ref_col]]
    .rename(columns={pred_col: 'predicted', ref_col: 'reference'})
    .dropna()
)
val_df['predicted'] = val_df['predicted'].astype(int)
val_df['reference'] = val_df['reference'].astype(int)
val_df = val_df[
    val_df['predicted'].isin([0,1]) &
    val_df['reference'].isin([0,1])
]

print(f"\n✓ Loaded {len(val_df):,} valid samples")
print(f"  Reference : {val_df['reference'].value_counts().sort_index().to_dict()}")
print(f"  Predicted : {val_df['predicted'].value_counts().sort_index().to_dict()}")

# ── Confusion Matrix & Metrics ─────────────────────────────────
y_true = val_df['reference'].values
y_pred = val_df['predicted'].values

cm = confusion_matrix(y_true, y_pred, labels=[0,1])
TN, FP, FN, TP = cm.ravel()

OA       = (TP+TN) / (TP+TN+FP+FN)
precision= TP/(TP+FP) if (TP+FP)>0 else 0.0
recall   = TP/(TP+FN) if (TP+FN)>0 else 0.0
f1       = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
kappa    = cohen_kappa_score(y_true, y_pred)
comm_err = FP/(TP+FP) if (TP+FP)>0 else 0.0
omit_err = FN/(TP+FN) if (TP+FN)>0 else 0.0

SEP = "─" * 65
print(f"\n{SEP}")
print("  TRIPLE-SOURCE VALIDATION RESULTS")
print("  Hansen GFC  +  ALOS PALSAR  +  MODIS Burned Area")
print(SEP)
print(f"  Overall Accuracy  : {OA:.4f}  ({OA*100:.2f}%)")
print(f"  Cohen's Kappa (κ) : {kappa:.4f}")
print(SEP)
print(f"  Precision         : {precision:.4f}")
print(f"  Recall            : {recall:.4f}")
print(f"  F1-Score          : {f1:.4f}")
print(SEP)
print(f"  Commission Error  : {comm_err:.4f}  ({comm_err*100:.2f}%)")
print(f"  Omission Error    : {omit_err:.4f}  ({omit_err*100:.2f}%)")
print(SEP)
print(f"  TP={TP:,}  TN={TN:,}  FP={FP}  FN={FN}")
print(SEP)
target_met = "✓  TARGET MET" if OA >= 0.85 else "✗  BELOW TARGET"
print(f"  Target ≥85% OA    : {target_met}")
print(SEP)
print(classification_report(y_true, y_pred,
      target_names=['Undisturbed','Disturbed'], digits=4, zero_division=0))

# Save
pd.DataFrame(list({
    'Overall Accuracy'       : round(OA,4),
    'Precision'              : round(precision,4),
    'Recall'                 : round(recall,4),
    'F1-Score'               : round(f1,4),
    "Cohen's Kappa"          : round(kappa,4),
    'Commission Error'       : round(comm_err,4),
    'Omission Error'         : round(omit_err,4),
    'TP':int(TP),'TN':int(TN),'FP':int(FP),'FN':int(FN),
    'Sources': 'Hansen GFC + ALOS PALSAR + MODIS Burned Area'
}.items()), columns=['Metric','Value']).to_csv('triple_validation_metrics.csv', index=False)
print("  Saved: triple_validation_metrics.csv")

# ── Figure ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Triple-Source Validation — SAR–Optical Disturbance Detection\n'
    'Hansen GFC + ALOS PALSAR + MODIS Burned Area  |  Pan-India Himalayan Region',
    fontsize=13, fontweight='bold'
)

# Confusion matrix
ax = axes[0]
cm_norm = np.nan_to_num(cm.astype(float)/cm.sum(axis=1,keepdims=True))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Proportion')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Undisturbed','Disturbed'], fontsize=11)
ax.set_yticklabels(['Undisturbed','Disturbed'], fontsize=11)
ax.set_xlabel('Predicted (SAR–Optical Model)', fontsize=11)
ax.set_ylabel('Reference (Triple-Source)', fontsize=11)
ax.set_title('Confusion Matrix — Triple-Source Reference', fontsize=12, fontweight='bold')
thresh = cm_norm.max()/2.0
for i in range(2):
    for j in range(2):
        c = 'white' if cm_norm[i,j]>thresh else 'black'
        ax.text(j,i,f'{cm_norm[i,j]:.3f}\n(n={cm[i,j]:,})',
                ha='center',va='center',fontsize=12,fontweight='bold',color=c)
ax.text(1.42,-0.45,f'OA={OA:.4f}\nκ={kappa:.4f}',
        ha='right',va='top',fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3',facecolor='lightyellow',edgecolor='gray'))

# Metrics bar
ax = axes[1]
bar_labels = ['Accuracy','Precision','Recall','F1-Score',"Kappa"]
bar_vals   = [OA, precision, recall, f1, kappa]
bar_colors = ['#2166ac','#4dac26','#d6604d','#8073ac','#762a83']
bars = ax.bar(bar_labels, bar_vals, color=bar_colors,
              edgecolor='white', linewidth=0.8, alpha=0.9)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Validation Metrics', fontsize=12, fontweight='bold')
ax.axhline(0.85, color='red', linestyle='--', linewidth=1.5, label='Target ≥85%')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, bar_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('triple_validation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: triple_validation_results.png")
print(f"\n{'='*65}")
print(f"  TRIPLE VALIDATION COMPLETE")
print(f"  OA={OA*100:.2f}%  |  κ={kappa:.4f}  |  F1={f1:.4f}")
print(f"  {'✓ TARGET ≥85% ACHIEVED' if OA>=0.85 else '⚠ Adjust thresholds in Cell 8'}")
print(f"{'='*65}")

---
## Validation Cell 18b — Hansen Forest Mask

In [ ]:
# ============================================================
# Validation Cell 18b — Hansen Forest Mask  (clip-first fix)
# ============================================================

val_hansen_forest = (
    tree_cover2000
    .clip(himalayan_aoi)          # clip before threshold
    .gte(30)
    .rename('hansen_forest')
    .selfMask()
)

hf_count = val_hansen_forest.reduceRegion(
    ee.Reducer.sum(), himalayan_aoi,
    scale=10000, maxPixels=1e8, bestEffort=True
).getInfo()

print('Hansen Forest Mask (treecover2000 ≥ 30%)')
print(f'  Valid pixels (10 km scale): {hf_count}')

---
## Validation Cell 18c — WorldCover Forest Mask

In [ ]:
# ============================================================
# Validation Cell 18c — ESA WorldCover Forest Mask
# ============================================================
# FIX: Removed .reproject(crs='EPSG:4326', scale=30)
#      .reproject() forces full materialisation at that scale
#      across the entire AOI → exceeds 2^31 pixel limit.
#      GEE applies the correct scale automatically at compute
#      time (reduceRegion / stratifiedSample), so no explicit
#      reproject is needed here.
# ============================================================

val_worldcover_forest = (
    worldcover_raw
    .eq(10)                    # class 10 = Trees (≥5 m canopy height)
    .clip(himalayan_aoi)
    .rename('worldcover_forest')
    .selfMask()
)

wc_count = val_worldcover_forest.reduceRegion(
    ee.Reducer.sum(), himalayan_aoi, 5000, maxPixels=1e8, bestEffort=True
).getInfo()

print('WorldCover Forest Mask (class 10 = Trees)')
print('  Resolution : 10 m native — no forced reproject')
print(f'  Valid pixels (5 km scale): {wc_count}')

---
## Validation Cell 18d — Dynamic World Forest Mask (Exclusion Only)

In [ ]:
# ============================================================
# Validation Cell 18d — Dynamic World Forest Mask
# ⚠ S2-derived — used as EXCLUSION MASK ONLY
# ============================================================
val_dw_forest = (
    dw_mode_image.eq(1)
    .clip(himalayan_aoi)
    .rename('dw_forest')
    .selfMask()
)
dw_count = val_dw_forest.reduceRegion(
    ee.Reducer.sum(), himalayan_aoi, 5000, maxPixels=1e8, bestEffort=True
).getInfo()
print('Dynamic World Forest Mask (class 1 = Trees)')
print('  ⚠ Role: Exclusion mask ONLY — S2-derived, not primary reference')
print(f'  Valid pixels (5 km scale): {dw_count}')


---
## Validation Cell 18e — Combined Triple-Source Forest Mask

In [ ]:
# ============================================================
# Validation Cell 18e — Combined Triple-Source Forest Mask
# Pixel enters validation only if Hansen AND WorldCover AND DW
# all agree it is forested.
# ============================================================
val_combined_forest = (
    val_hansen_forest
    .And(val_worldcover_forest)
    .And(val_dw_forest)
    .clip(himalayan_aoi)
    .rename('combined_forest')
    .selfMask()
)
combined_count = val_combined_forest.reduceRegion(
    ee.Reducer.sum(), himalayan_aoi, 5000, maxPixels=1e8, bestEffort=True
).getInfo()
print('Combined Triple-Source Forest Mask')
print('  Logic : Hansen ≥30% AND WorldCover class-10 AND DW class-1')
print(f'  Valid pixels (5 km scale): {combined_count}')


---
## Validation Cell 18f — Hansen Disturbance Reference

In [ ]:
# ============================================================
# Validation Cell 18f — Hansen Disturbance Reference (2018–2024)
# ============================================================
# FIX: .clip() BEFORE .unmask(0)
#   Old order: loss_year → gte/lte → unmask(0) [global] → clip → timeout
#   New order: loss_year → clip → gte/lte → unmask(0) [AOI only] → fast
#
# Also: updateMask(val_combined_forest) already restricts pixels,
# so unmask(0) only fills within the forest mask extent.
# Stats check raised to scale=10000 for faster .getInfo().
# ============================================================

val_hansen_reference = (
    loss_year
    .clip(himalayan_aoi)          # clip FIRST — restrict to AOI before any fill
    .gte(18).And(
        loss_year.clip(himalayan_aoi).lte(24)
    )
    .unmask(0)                    # now fills only within AOI extent
    .updateMask(val_combined_forest)
    .rename('hansen_reference')
    .toByte()
)

# Use 10 km scale for stats — fast, sufficient for a histogram check
ref_stats = val_hansen_reference.reduceRegion(
    reducer    = ee.Reducer.frequencyHistogram(),
    geometry   = himalayan_aoi,
    scale      = 10000,           # raised from 5000 → 10000 for speed
    maxPixels  = 1e8,
    bestEffort = True
).getInfo()

ref_hist  = ref_stats.get('hansen_reference', {})
undist_px = int(float(ref_hist.get('0', ref_hist.get(0, 0))))
dist_px   = int(float(ref_hist.get('1', ref_hist.get(1, 0))))
total_px  = undist_px + dist_px

print('Hansen Disturbance Reference (2018–2024)')
print('  Period    : lossyear 18–24 (Landsat-based, sensor-independent)')
print(f'  0 — Undisturbed : {undist_px:>8,} pixels')
print(f'  1 — Disturbed   : {dist_px:>8,} pixels  ({dist_px/max(total_px,1)*100:.2f}%)')
print(f'  Total forest    : {total_px:>8,} pixels')

---
## Validation Cell 18g — Stratified Validation Sampling

In [ ]:
# ============================================================
# Validation Cell 18g — Part 1: Export Samples to Drive
# ============================================================
# Pulling 10,000 points via .getInfo() times out on large AOIs.
# Solution: export to CSV via Drive, load locally in Part 2.
#
# Run this cell, wait for the task to complete, then run Part 2.
# Monitor: https://code.earthengine.google.com/tasks
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score
import numpy as np

N_SAMPLES_PER_CLASS = 3000    # 3000 × 2 = 6000 total (reduced to avoid timeout)
VAL_SCALE           = 30
VAL_SEED            = 42

# ── Model prediction binary map ───────────────────────────────
val_predicted = (
    disturbance_year_map
    .clip(himalayan_aoi)
    .gt(0)
    .unmask(0)
    .updateMask(val_combined_forest)
    .rename('predicted')
    .toByte()
)

# ── Combine predicted + reference into 2-band image ──────────
val_image = val_predicted.addBands(val_hansen_reference)

# ── Stratified sample ─────────────────────────────────────────
print(f'Generating stratified sample: {N_SAMPLES_PER_CLASS} pts per class at {VAL_SCALE} m ...')

val_samples = val_image.stratifiedSample(
    numPoints   = N_SAMPLES_PER_CLASS,
    classBand   = 'hansen_reference',
    region      = himalayan_aoi,
    scale       = VAL_SCALE,
    seed        = VAL_SEED,
    tileScale   = 8,           # increased from 4 → 8 for large AOI
    geometries  = False
)

# ── Export to Google Drive ────────────────────────────────────
export_val = ee.batch.Export.table.toDrive(
    collection     = val_samples,
    description    = 'himalayan_validation_samples',
    folder         = 'GEE_Exports',
    fileNamePrefix = 'himalayan_validation_samples',
    fileFormat     = 'CSV',
    selectors      = ['predicted', 'hansen_reference']
)
export_val.start()

print(f'  Export task submitted')
print(f'  Task ID : {export_val.id}')
print()
print('━' * 60)
print('NEXT STEP:')
print('  1. Wait for task to complete:')
print('     https://code.earthengine.google.com/tasks')
print('  2. Download himalayan_validation_samples.csv from')
print('     Google Drive → GEE_Exports folder')
print('  3. Place the CSV in the same folder as this notebook')
print('  4. Run Part 2 (next cell)')
print('━' * 60)

In [ ]:
# ── DIAGNOSTIC — run this to see exact CSV columns ────────────
import pandas as pd
df_check = pd.read_csv('himalayan_validation_samples.csv')
print('All columns:')
for col in df_check.columns:
    print(f'  repr: {repr(col)}')
print('\nFirst 3 rows:')
print(df_check.head(3).to_string())

In [ ]:
# ============================================================
# Validation Cell 18g — Part 2: Load CSV & Prepare val_df
# ============================================================
# Columns confirmed: 'predicted' and 'hansen_reference'
# ============================================================

import os
import pandas as pd

CSV_PATH = 'himalayan_validation_samples.csv'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f'{CSV_PATH} not found.\n'
        f'Files present: {os.listdir(".")}'
    )

raw_df = pd.read_csv(CSV_PATH)

# Direct rename — columns confirmed as 'predicted' and 'hansen_reference'
val_df = (
    raw_df[['predicted', 'hansen_reference']]
    .rename(columns={'hansen_reference': 'reference'})
    .dropna()
)

val_df['predicted'] = val_df['predicted'].astype(int)
val_df['reference'] = val_df['reference'].astype(int)

# Keep only valid binary values (0 or 1)
val_df = val_df[val_df['predicted'].isin([0, 1]) & val_df['reference'].isin([0, 1])]

print(f'✓ val_df ready')
print(f'  Total rows        : {len(val_df):,}')
print(f'  Reference dist    : {val_df["reference"].value_counts().sort_index().to_dict()}')
print(f'  Predicted dist    : {val_df["predicted"].value_counts().sort_index().to_dict()}')

---
## Validation Cell 18h — Confusion Matrix

In [ ]:
# ============================================================
# Validation Cell 18h — Confusion Matrix
# TP = model disturbed AND Hansen disturbed
# TN = model undisturbed AND Hansen undisturbed
# FP = model disturbed BUT Hansen undisturbed (commission)
# FN = model undisturbed BUT Hansen disturbed (omission)
# ============================================================

y_true = val_df['reference'].values
y_pred = val_df['predicted'].values

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
TN, FP, FN, TP = cm.ravel()

cm_df = pd.DataFrame(
    cm,
    index   = pd.Index(['Hansen: Undisturbed (0)', 'Hansen: Disturbed (1)'], name='Reference'),
    columns = pd.Index(['Model: Undisturbed (0)', 'Model: Disturbed (1)'],   name='Predicted')
)

print('=' * 60)
print('CONFUSION MATRIX — SAR–Optical vs Hansen GFC Reference')
print('=' * 60)
print(cm_df.to_string())
print()
print(f'  TP : {TP:>6,}')
print(f'  TN : {TN:>6,}')
print(f'  FP : {FP:>6,}  [commission error]')
print(f'  FN : {FN:>6,}  [omission error]')
print(f'  Total : {TP+TN+FP+FN:>6,}')


---
## Validation Cell 18i — Accuracy Metrics

In [ ]:
# ============================================================
# Validation Cell 18i — Accuracy Metrics
# ============================================================

OA        = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)  if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN)  if (TP + FN) > 0 else 0.0
f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0
kappa     = cohen_kappa_score(y_true, y_pred)

commission_err = FP / (TP + FP) if (TP + FP) > 0 else 0.0
omission_err   = FN / (TP + FN) if (TP + FN) > 0 else 0.0

report = classification_report(y_true, y_pred,
    target_names=['Undisturbed', 'Disturbed'], digits=4, zero_division=0)

val_metrics = {
    'Overall Accuracy'      : round(OA, 4),
    'Precision (Disturbed)' : round(precision, 4),
    'Recall (Disturbed)'    : round(recall, 4),
    'F1-Score (Disturbed)'  : round(f1, 4),
    "Cohen's Kappa"         : round(kappa, 4),
    'Commission Error'      : round(commission_err, 4),
    'Omission Error'        : round(omission_err, 4),
    'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN)
}

SEP = '─' * 50
print(SEP)
print('  Validation Results')
print('  SAR–Optical Model vs Hansen GFC Reference')
print('  Pan-India Himalayan Region | 2018–2024')
print(SEP)
print(f'  Overall Accuracy  : {OA:.4f}  ({OA*100:.2f} %)')
print(f"  Cohen's Kappa (κ) : {kappa:.4f}")
print(SEP)
print(f'  Precision         : {precision:.4f}')
print(f'  Recall            : {recall:.4f}')
print(f'  F1-Score          : {f1:.4f}')
print(SEP)
print(f'  Commission Error  : {commission_err:.4f}  ({commission_err*100:.2f} %)')
print(f'  Omission Error    : {omission_err:.4f}  ({omission_err*100:.2f} %)')
print(SEP)
print('  Per-Class Report:')
print(report)
print(SEP)
print('  κ > 0.80 → Strong  |  0.61–0.80 → Substantial  |  OA > 85% → Publication-grade')
print(SEP)

pd.DataFrame(list(val_metrics.items()), columns=['Metric','Value']
             ).to_csv('validation_metrics.csv', index=False)
print('  Saved: validation_metrics.csv')


---
## Validation Cell 18j — Visualisation

In [ ]:
# ============================================================
# Validation Cell 18j — Visualisation
# ============================================================
# Figure 1: IEEE-style confusion matrix
# Figure 2: Metrics bar + TP/TN/FP/FN pie
# Figure 3: geemap predicted vs reference comparison
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Figure 1: Confusion Matrix (IEEE style) ───────────────────
fig1, ax1 = plt.subplots(figsize=(7, 6))

cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))

im = ax1.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
fig1.colorbar(im, ax=ax1, fraction=0.046, pad=0.04, label='Proportion')

ax1.set_xticks([0, 1])
ax1.set_yticks([0, 1])
ax1.set_xticklabels(['Undisturbed', 'Disturbed'], fontsize=12)
ax1.set_yticklabels(['Undisturbed', 'Disturbed'], fontsize=12)
ax1.set_xlabel('Predicted (SAR–Optical Model)', fontsize=12, labelpad=8)
ax1.set_ylabel('True Label (Hansen GFC)', fontsize=12, labelpad=8)
ax1.set_title(
    'Confusion Matrix — Forest Disturbance Detection\n'
    'Pan-India Himalayan Region  |  2018–2024',
    fontsize=12, fontweight='bold', pad=12
)

thresh = cm_norm.max() / 2.0
for i in range(2):
    for j in range(2):
        color = 'white' if cm_norm[i, j] > thresh else 'black'
        ax1.text(
            j, i,
            f'{cm_norm[i, j]:.3f}\n(n={cm[i, j]:,})',
            ha='center', va='center',
            fontsize=13, fontweight='bold', color=color
        )

ax1.set_xlim(-0.5, 1.5)
ax1.set_ylim(1.5, -0.5)

# Extract metric values before f-string to avoid quote conflict
oa_val    = val_metrics['Overall Accuracy']
kappa_val = val_metrics["Cohen's Kappa"]
annot_text = f'OA = {oa_val:.4f}\n\u03ba  = {kappa_val:.4f}'

ax1.text(
    1.42, -0.45, annot_text,
    ha='right', va='top', fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')
)

plt.tight_layout()
plt.savefig('validation_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: validation_confusion_matrix.png')


# ── Figure 2: Metrics Dashboard ───────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle(
    'Validation Metrics — SAR–Optical Disturbance Detection\n'
    'Pan-India Himalayan Region  |  2018–2024',
    fontsize=13, fontweight='bold'
)

# Left panel: main metrics bar chart
ax = axes2[0]

prec_val  = val_metrics['Precision (Disturbed)']
rec_val   = val_metrics['Recall (Disturbed)']
f1_val    = val_metrics['F1-Score (Disturbed)']

bar_labels = ['Overall\nAccuracy', 'Precision', 'Recall', 'F1-Score', "Cohen's\nKappa"]
bar_values = [oa_val, prec_val, rec_val, f1_val, kappa_val]
bar_colors = ['#2166ac', '#4dac26', '#d6604d', '#8073ac', '#762a83']

bars = ax.bar(
    bar_labels, bar_values,
    color=bar_colors, edgecolor='white', linewidth=0.8, alpha=0.9
)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Primary Accuracy Metrics', fontsize=11, pad=8)
ax.axhline(0.80, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='0.80 reference')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)

for bar, val in zip(bars, bar_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f'{val:.4f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

# Right panel: TP / TN / FP / FN composition pie
ax = axes2[1]

total       = TP + TN + FP + FN
pie_labels  = ['TP', 'TN', 'FP\n(Commission)', 'FN\n(Omission)']
pie_values  = [TP / total, TN / total, FP / total, FN / total]
pie_colors  = ['#1a9641', '#a6d96a', '#d7191c', '#fdae61']

wedges, texts, autotexts = ax.pie(
    pie_values,
    labels=pie_labels,
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')

ax.set_title('Sample Composition\n(TP / TN / FP / FN)', fontsize=11, pad=8)

count_text = f'TP={TP:,}  TN={TN:,}\nFP={FP:,}  FN={FN:,}'
ax.text(
    0, -1.5, count_text,
    ha='center', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.8)
)

plt.tight_layout()
plt.savefig('validation_metrics_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: validation_metrics_dashboard.png')


# ── Figure 3: geemap — predicted vs reference comparison ──────
Map_v = geemap.Map()
Map_v.centerObject(himalayan_aoi, zoom=6)

Map_v.addLayer(
    val_predicted,
    {'min': 0, 'max': 1, 'palette': ['white', '#bd0026']},
    'Model: Predicted Disturbance'
)
Map_v.addLayer(
    val_hansen_reference,
    {'min': 0, 'max': 1, 'palette': ['white', '#1a9641']},
    'Reference: Hansen Loss 2018–2024'
)
Map_v.addLayer(
    val_combined_forest.visualize(min=0, max=1, palette=['#d4e6b5']),
    {},
    'Triple-Source Forest Mask', False
)
Map_v.addLayer(
    ee.Image().paint(ee.FeatureCollection(himalayan_aoi), 0, 2),
    {'palette': ['black']},
    'AOI Boundary'
)
Map_v.add_legend(
    title='Validation Layer',
    keys=['Model Disturbed', 'Hansen Loss', 'Forest Mask'],
    colors=['#bd0026', '#1a9641', '#d4e6b5']
)

print('\nMap: Predicted vs Reference Disturbance Comparison')
Map_v

---
## Cell 19 — Export Final Products to GEE Assets

In [ ]:
# ============================================================
# Cell 19 — Export Final Analysis Products to GEE Assets
# ============================================================

def export_to_asset(image, name, description, scale=100):
    task = ee.batch.Export.image.toAsset(
        image=image.toFloat(), description=description,
        assetId=f'{ASSET_DIR}/{name}',
        region=himalayan_aoi, scale=scale, maxPixels=1e11, crs='EPSG:4326'
    )
    task.start()
    print(f'  {name:<35} Task: {task.id}')
    return task

print(f'Exporting final products → {ASSET_DIR}\n')
export_to_asset(disturbance_year_map,  'disturbance_year_map',  'disturbance_year_map')
export_to_asset(disturbance_frequency, 'disturbance_frequency', 'disturbance_frequency')
export_to_asset(hotspot_map,           'hotspot_map',           'hotspot_map')
export_to_asset(recovery_rate_map,     'recovery_rate_map',     'recovery_rate_map')
export_to_asset(sar_recovery_map,      'sar_recovery_map',      'sar_recovery_map')
export_to_asset(recovery_class_map,    'recovery_class_map',    'recovery_class_map')
export_to_asset(recovery_time_map,     'recovery_time_map',     'recovery_time_map')
print('\n✓ All export tasks submitted.')
print('Monitor: https://code.earthengine.google.com/tasks')


---
## Cell 20 — Output Variable Registry

In [ ]:
# ============================================================
# Cell 20 — Output Variable Registry
# ============================================================

registry = [
    ('annual_optical',         'ee.ImageCollection', 'S2 SR composites [NDVI, NBR, NDMI] 2017–2024'),
    ('annual_sar',             'ee.ImageCollection', 'S1 SAR composites [VH, VV_VH_ratio] 2017–2024'),
    ('fused_composites',       'ee.ImageCollection', 'Fused [NDVI, NBR, NDMI, VH, VV_VH_ratio] 2017–2024'),
    ('annual_disturbances',    'ee.ImageCollection', 'Annual disturbance masks 2018–2024'),
    ('disturbance_year_map',   'ee.Image',           'First disturbance year per pixel (Int16, 2018–2024)'),
    ('disturbance_frequency',  'ee.Image',           'Count of disturbance events per pixel (0–7)'),
    ('hotspot_map',            'ee.Image',           '1=Low, 2=Moderate, 3=High disturbance'),
    ('recovery_rate_map',      'ee.Image',           'NDVI slope b (yr⁻¹) per disturbed pixel'),
    ('sar_recovery_map',       'ee.Image',           'VH backscatter slope (dB yr⁻¹)'),
    ('recovery_time_map',      'ee.Image',           'Estimated years to return to pre-disturbance NDVI'),
    ('recovery_class_map',     'ee.Image',           '1=None, 2=Slow, 3=Moderate, 4=Fast recovery'),
]

w = [26, 20, 55]
sep = '─' * (sum(w) + 4)
print(sep)
print('OUTPUT VARIABLE REGISTRY')
print(sep)
print(f'{"Variable":<{w[0]}} {"Type":<{w[1]}} {"Description"}')
print(sep)
for var, typ, desc in registry:
    print(f'{var:<{w[0]}} {typ:<{w[1]}} {desc}')
print(sep)
print('\nPipeline complete.')


---
## Pipeline Reference

| Cell | Stage | Key Detail | Output |
|------|-------|-----------|--------|
| 1 | Setup | GEE auth, imports, constants | — |
| 2 | GADM Diagnostic | Inspect zip contents | — |
| 3 | Study Area | Local GADM 4.1 → AOI, Hansen forest mask, SRTM | `himalayan_aoi`, `forest_mask` |
| 4 | Preprocessing fns | Cloud mask, snow mask, vegetation indices | functions |
| 5 | Composite builders | Standard (2019–2024) + Robust (2017–2018) | functions |
| **6** | **⬆ EXPORT** | Build + export fused composites → GEE Assets | `fused_YYYY` |
| **7** | **⬇ LOAD** | Reload flat assets + coverage check | `annual_optical`, `annual_sar`, `fused_composites` |
| 8 | Detection fn | Bi-temporal SAR–Optical change detection | `compute_disturbance()` |
| 9 | Annual disturbances | Apply detection 2018–2024 | `annual_disturbances` |
| 10 | Year map | `assign_year()` + `.min()` with `.toInt16()` | `disturbance_year_map` |
| 11 | Frequency | `.sum()` over 7 years | `disturbance_frequency` |
| 12 | Hotspots | 3-class threshold | `hotspot_map` |
| 13 | Recovery model | `linearFit()` on NDVI + VH | `recovery_rate_map`, `sar_recovery_map`, `recovery_time_map` |
| 14 | Recovery class | 4-class NDVI slope threshold | `recovery_class_map` |
| 15 | geemap maps | 5 interactive maps | — |
| 16 | matplotlib | 3-panel summary figure | `himalayan_disturbance_summary.png` |
| 17 | Elevation analysis | Disturbance by elevation zone | `elevation_df` |
| 18a–j | Validation | Multi-source validation (Hansen+WorldCover+DW) | `validation_metrics.csv` |
| 19 | Export products | 7 final maps → GEE Assets | — |
| 20 | Registry | Output variable summary | — |

---
### All Fixes Applied

| # | Issue | Fix |
|---|-------|-----|
| 1 | `GADM/level1` asset not found | Read local `gadm41_IND_shp.zip` via geopandas |
| 2 | `User memory limit exceeded` | Two-phase workflow: export composites as Assets, reload |
| 3 | `Short<0,2018>` type mismatch | `.toInt16()` explicit cast in `assign_year()` |
| 4 | `getInfo()` memory limit | `scale=5000` + `bestEffort=True` on all verification calls |
| 5 | 2017/2018 sparse composites | Robust builder: Apr–Nov window, 85% cloud cap, mean reducer |
| 6 | Hotspot + recovery charts empty | `get_hist_val()` tries `'1'`, `'1.0'`, `1` key formats |
